# Preparação dos dados do ONS

Este notebook transforma o arquivo de interrupções de carga do ONS em uma tabela com uma linha por dia para o Subsistema Sul. O arquivo original do portal foi usado como base, sem nenhuma agregação já feita, e o resultado é salvo em data/interim para ser combinado com o clima e os feriados no notebook de integração. Os indicadores CIPER e SM não são considerados porque são mensais e anuais, e só servem para a análise sazonal da segunda pergunta de pesquisa.

In [ ]:
from pathlib import Path
from datetime import datetime
import urllib.request

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

RAW = Path("../data/raw/ONS")
INTERIM = Path("../data/interim")
ARQUIVO = RAW / "INTERRUPCAO_CARGA.csv"
URL = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/interrupcao_carga/INTERRUPCAO_CARGA.csv"

INICIO = "2021-01-01"
FIM = "2025-12-31"

### Leitura do dataset

In [ ]:
interrupcoes = duckdb.sql(f"""
    SELECT cod_perturbacao AS perturbacao,
           din_interrupcaocarga AS horario,
           id_subsistema AS subsistema,
           id_estado AS estado,
           nom_agente AS agente,
           val_cargainterrompida_mw AS carga_mw,
           flg_envolveuredebasica AS rede_basica
    FROM read_csv_auto('{ARQUIVO}', delim=';', header=true)
""").df()

print(len(interrupcoes), "linhas")
for coluna in ["subsistema", "estado", "rede_basica"]:
    print(interrupcoes[coluna].value_counts(dropna=False).head(10), "\n")

### Datas e valores numéricos

A coluna de horário é convertida para data e hora, e as linhas em que isso falha são contadas e depois descartadas. O dia de cada evento é a data desse horário, considerando o horário de Brasília.

In [ ]:
interrupcoes["horario"] = pd.to_datetime(interrupcoes["horario"], errors="coerce")
print("datas inválidas:", interrupcoes["horario"].isna().sum())

interrupcoes = interrupcoes.dropna(subset=["horario"]).copy()
interrupcoes["data"] = interrupcoes["horario"].dt.normalize()
interrupcoes["carga_mw"] = pd.to_numeric(interrupcoes["carga_mw"].astype(str).str.replace(",", "."), errors="coerce")

### Duplicados

Como o código da perturbação se repete legitimamente, ele sozinho não é muito útil para achar campos duplicados. Então, foram removidas apenas as linhas completamente idênticas e depois verificamos se sobraram registros com a mesma perturbação, agente, estado e horário, o que indicaria um reenvio com valores corrigidos e exigiria uma decisão caso a caso.

In [ ]:
antes = len(interrupcoes)
interrupcoes = interrupcoes.drop_duplicates()
print("linhas idênticas removidas:", antes - len(interrupcoes))

chave = ["perturbacao", "agente", "estado", "horario"]
print("registros com a mesma chave:", interrupcoes.duplicated(chave, keep=False).sum())

### Recorte

Mantemos só o Subsistema Sul e o período de análise. 

In [ ]:
sul = interrupcoes[(interrupcoes["subsistema"] == "S") & interrupcoes["data"].between(INICIO, FIM)].copy()

print(len(sul), "linhas no recorte")
print(sul["estado"].value_counts())

### Rede Básica

Antes de decidir se a Rede Básica será a referência para a filtragem, confirmamos que proporção dos registros seria descartada. Se for pequena, o filtro quase não muda nada. Se for grande, ele muda o que estamos chamando de interrupção, e a escolha precisa ser justificada no relatório. A decisão fica no parâmetro abaixo.

In [ ]:
sul["rede_basica"] = sul["rede_basica"].astype(str).str.upper().isin(["1", "S", "SIM", "TRUE"])
print(sul["rede_basica"].value_counts(normalize=True).round(3))

SO_REDE_BASICA = False
if SO_REDE_BASICA:
    sul = sul[sul["rede_basica"]].copy()

### Valores suspeitos

Registros sem carga, com carga zero ou com carga negativa não servem para medir a magnitude de uma interrupção, então são descartados depois de contados. Os valores muito altos, por outro lado, são mantidos. Interrupções grandes e reais são justamente o que o modelo precisa aprender a prever, e remover outliers aqui esconderia os eventos de interesse. Olhamos a distribuição e os dez maiores registros apenas para garantir que nenhum deles é um erro evidente de digitação.

In [ ]:
print("sem carga:", sul["carga_mw"].isna().sum(), "| zero:", (sul["carga_mw"] == 0).sum(), "| negativa:", (sul["carga_mw"] < 0).sum())
sul = sul[sul["carga_mw"] > 0]

sul["carga_mw"].plot.hist(bins=50, log=True, title="Carga interrompida por registro (MW, escala log)")
plt.show()
sul.nlargest(10, "carga_mw")[["perturbacao", "horario", "estado", "agente", "carga_mw"]]

### Agregação diária

Cada dia vira uma linha com duas medidas: quantas perturbações distintas ocorreram, contando códigos únicos para não inflar o número quando uma perturbação atinge vários agentes, e quantos megawatts foram interrompidos no total. Os dias sem nenhum registro recebem zero em vez de valor ausente. A descrição do portal menciona um critério de 100 MW por 10 minutos, o que indica que o conjunto guarda só interrupções relevantes. Um dia sem registro significa então que não houve interrupção relevante, e não que o dado deixou de ser coletado. É por isso que reindexamos para um calendário completo preenchendo com zero.

In [ ]:
calendario = pd.date_range(INICIO, FIM, freq="D", name="data")

por_dia = (
    sul.groupby("data")
       .agg(n_perturbacoes=("perturbacao", "nunique"), mw_interrompido=("carga_mw", "sum"))
       .reindex(calendario, fill_value=0)
       .reset_index()
)

print(len(por_dia), "dias, sendo", (por_dia["n_perturbacoes"] > 0).sum(), "com pelo menos uma perturbação")
por_dia.set_index("data")["mw_interrompido"].plot(figsize=(12, 3), title="MW interrompidos por dia no Sul")
plt.show()

### Exportação

A série tem exatamente um registro por dia do período, o que a verificação abaixo confirma antes de salvar. A variável-alvo de alta interrupção não é criada aqui. O limiar, por exemplo um percentil dos MW diários, deve ser calculado só com os dados de treino, e defini-lo agora faria informação do período de teste vazar para o modelo.

In [ ]:
assert por_dia["data"].is_unique and len(por_dia) == len(calendario)
por_dia.to_parquet(INTERIM / "ons_diario_sul.parquet", index=False)